In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_excel('HR_data.xlsx')

In [ ]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [ ]:
len(df)

1470

In [ ]:
len(df.columns)

35

## Age Band

In [ ]:
df['Age_Band'] = pd.cut(
    df['Age'],
    bins=[18,25,35,45,55,65],
    labels=[
        '18-25',
        '26-35',
        '36-45',
        '46-55',
        '56-65'
    ]
)

## Income Bnad

In [ ]:
df['Income_Band'] = pd.qcut(
    df['MonthlyIncome'],
    q=4,
    labels=[
        'Low',
        'Medium',
        'High',
        'Very High'
    ]
)

## Tenure Band

In [ ]:
df['Tenure_Band'] = pd.cut(
    df['YearsAtCompany'],
    bins=[-1,1,3,5,100],
    labels=[
        'New Hire',
        'Early Career',
        'Mid Career',
        'Long Tenure'
    ]
)

## Promotion Gap

In [ ]:
df['Promotion_Gap'] = pd.cut(
    df['YearsSinceLastPromotion'],
    bins=[-1,1,3,5,100],
    labels=[
        'Recently Promoted',
        '2-3 Years',
        '4-5 Years',
        '5+ Years'
    ]
)

## Company's Attrition

In [ ]:
company_attrition = (
    (df['Attrition'] == 'Yes')
    .mean()
) * 100

print(
    f"Company Attrition Rate: {company_attrition:.2f}%"
)

Company Attrition Rate: 16.12%


## Metrics & Definitions

### Attrition Rate

Measures the percentage of employees who left within a specific segment.

Formula:

Attrition Rate (%) =
(Number of Employees Who Left ÷ Total Employees in Segment) × 100

**Example:**

Sales Department:
- Employees = 446
- Exits = 92

Attrition Rate = (92 ÷ 446) × 100 = 20.63%

---

### Company Average Attrition Rate

Measures the overall attrition rate across the organization.

Formula:

Company Attrition Rate (%) =
(Total Exits ÷ Total Employees) × 100

This serves as the baseline for comparison.

---

### Difference from Company Average

Shows how far a segment's attrition rate is above or below the company average.

Formula:

Difference =
Segment Attrition Rate − Company Average Attrition Rate

**Interpretation:**
- Positive value → Higher attrition than company average.
- Negative value → Lower attrition than company average.

---

### Attrition Index

Measures the relative attrition risk of a segment compared to the company average.

Formula:

Attrition Index =
(Segment Attrition Rate ÷ Company Average Attrition Rate) × 100

**Interpretation:**
- 100 = Equal to company average.
- >100 = Higher attrition risk than average.
- <100 = Lower attrition risk than average.

**Example:**

Segment Attrition Rate = 32%
Company Average = 16%

Attrition Index = (32 ÷ 16) × 100 = 200

This means the segment has 2x the attrition risk of the average employee.

---

### Contribution %

Measures the share of total exits contributed by a segment within a dimension.

Formula:

Contribution (%) =
(Segment Exits ÷ Total Exits in the Dimension) × 100

**Example:**

Department Dimension:
- Sales Exits = 92
- Total Department Exits = 237

Contribution = (92 ÷ 237) × 100 = 38.82%

**Interpretation:**
A higher contribution indicates that the segment accounts for a larger proportion of employee exits.

> Note: A segment may have a high contribution because it contains a large employee population, not necessarily because it has the highest attrition rate. Therefore, Contribution Analysis should be interpreted alongside Attrition Rate Analysis.

---

### Highest Risk Segment

The segment with the highest Attrition Rate within a dimension.

**Example:**

Job Role Dimension:

- Sales Representative → 39.76%
- Laboratory Technician → 23.94%
- Research Scientist → 16.10%

Highest Risk Segment = Sales Representative

---

### Largest Contributor

The segment contributing the highest number of exits within a dimension.

**Example:**

Job Role Dimension:

- Laboratory Technician → 62 exits
- Sales Representative → 33 exits

Largest Contributor = Laboratory Technician

Although Sales Representatives have a higher attrition rate, Laboratory Technicians contribute more exits due to their larger employee population.

### attrition_analysis()

This function performs a detailed attrition analysis for a single column (e.g., Department, Job Role, Overtime).

It calculates:
- Total employees
- Total attritions
- Attrition rate
- Company average attrition rate
- Difference from company average
- Attrition Index

**Why it is useful:**
- Helps identify which groups have higher attrition than the company average.
- Provides a detailed view of attrition patterns within a specific dimension.
- Serves as the foundation for deeper analysis and business insights.

In [ ]:
def attrition_analysis(
    df,
    column,
    target='Attrition',
    positive_class='Yes',
    min_population=20
):

    company_avg = (
        (df[target] == positive_class)
        .mean()
    ) * 100

    summary = df.groupby(column).agg(

        Employees=(target,'count'),

        Exits=(
            target,
            lambda x:
            (x==positive_class).sum()
        )

    )

    summary['Attrition_Rate'] = (

        summary['Exits']
        /
        summary['Employees']

    ) * 100

    summary = summary[
        summary['Employees']
        >= min_population
    ]

    summary['Company_Avg'] = company_avg

    summary['Difference'] = (

        summary['Attrition_Rate']
        -
        company_avg

    )

    summary['Attrition_Index'] = (

        summary['Attrition_Rate']
        /
        company_avg

    ) * 100

    return summary.sort_values(
        'Attrition_Rate',
        ascending=False
    ).round(2)

### attrition_driver_scan()

This function scans multiple columns and identifies the highest-risk segment within each dimension.

For example:
- Highest-risk Department
- Highest-risk Job Role
- Highest-risk Age Band

**Why it is useful:**
- Quickly highlights the strongest attrition drivers.
- Eliminates the need to manually analyze every column.
- Helps focus further investigation on the most vulnerable employee groups.

In [ ]:
def attrition_driver_scan(
    df,
    columns,
    target='Attrition',
    positive_class='Yes'
):

    findings = []

    for col in columns:

        result = attrition_analysis(
            df,
            col,
            target,
            positive_class
        )

        if len(result) > 0:

            highest = result.iloc[0]

            findings.append({

                'Dimension': col,

                'Highest Risk Segment':
                    highest.name,

                'Employees':
                    highest['Employees'],

                'Exits':
                    highest['Exits'],

                'Attrition Rate':
                    highest['Attrition_Rate'],

                'Attrition Index':
                    highest['Attrition_Index']

            })

    findings = pd.DataFrame(findings)

    return findings.sort_values(
        'Attrition Rate',
        ascending=False
    )

### contribution_scan()

This function identifies the segment that contributes the highest number of employee exits within each dimension.

Contribution % is calculated as:

Contribution % =
(Number of exits from a segment ÷ Total exits in that dimension) × 100

For example, in the Department dimension:

Contribution % =
(Exits from Sales Department ÷ Total exits across all Departments) × 100

**Why it is useful:**
- Shows which segments generate the largest share of attrition.
- Helps distinguish between high-risk groups and high-impact groups.
- Supports prioritization by identifying where attrition affects the largest number of employees.
- Helps HR focus retention efforts on areas with the greatest overall business impact.

In [ ]:
def contribution_scan(
    df,
    columns,
    target='Attrition',
    positive_class='Yes'
):

    findings = []

    for col in columns:

        summary = df.groupby(col).agg(

            Employees=(target,'count'),

            Exits=(
                target,
                lambda x:
                (x==positive_class).sum()
            )
        )

        summary['Contribution_%'] = (

            summary['Exits']
            /
            summary['Exits'].sum()

        ) * 100

        summary['Attrition_Rate'] = (

        summary['Exits']
        /
        summary['Employees']
       ) * 100

        highest = summary.sort_values(
            'Contribution_%',
            ascending=False
        ).iloc[0]



        findings.append({

            'Dimension': col,

            'Largest Contributor':
                highest.name,
            'Contributor_Employees':
                highest['Employees'],

            'Contributor_Exits':
                highest['Exits'],

            'Contribution_%':
                round(
                    highest['Contribution_%'],
                    2
                ),
                'Contributor Attrition Rate':
    round(
        highest['Attrition_Rate'],
        2
    )

        })

    return pd.DataFrame(findings)

In [ ]:
driver_columns = ['Department','JobRole','BusinessTravel','Gender','MaritalStatus',
                  'EducationField','OverTime','JobLevel','Age_Band','Income_Band',
                  'Tenure_Band','Promotion_Gap','JobSatisfaction','EnvironmentSatisfaction',
                  'RelationshipSatisfaction','WorkLifeBalance','StockOptionLevel']

In [ ]:
driver_scan = attrition_driver_scan(
    df,
    driver_columns
)

driver_scan

,Dimension,Highest Risk Segment,Employees,Exits,Attrition Rate,Attrition Index
1,JobRole,Sales Representative,83.0,33.0,39.76,246.61
10,Tenure_Band,New Hire,215.0,75.0,34.88,216.37
8,Age_Band,18-25,115.0,40.0,34.78,215.74
15,WorkLifeBalance,1,80.0,25.0,31.25,193.83
6,OverTime,Yes,416.0,127.0,30.53,189.36
9,Income_Band,Low,369.0,108.0,29.27,181.54
7,JobLevel,1,543.0,143.0,26.34,163.34
5,EducationField,Human Resources,27.0,7.0,25.93,160.81
4,MaritalStatus,Single,470.0,120.0,25.53,158.36
13,EnvironmentSatisfaction,1,284.0,72.0,25.35,157.25


In [ ]:
contribution_scan(
    df,
    driver_columns
)

,Dimension,Largest Contributor,Contributor_Employees,Contributor_Exits,Contribution_%,Contributor Attrition Rate
0,Department,Research & Development,961.0,133.0,56.12,13.84
1,JobRole,Laboratory Technician,259.0,62.0,26.16,23.94
2,BusinessTravel,Travel_Rarely,1043.0,156.0,65.82,14.96
3,Gender,Male,882.0,150.0,63.29,17.01
4,MaritalStatus,Single,470.0,120.0,50.63,25.53
5,EducationField,Life Sciences,606.0,89.0,37.55,14.69
6,OverTime,Yes,416.0,127.0,53.59,30.53
7,JobLevel,1,543.0,143.0,60.34,26.34
8,Age_Band,26-35,606.0,116.0,49.79,19.14
9,Income_Band,Low,369.0,108.0,45.57,29.27


In [ ]:
risk_matrix = driver_scan.merge(

    contribution_scan(
        df,
        driver_columns
    ),

    on='Dimension'

)

risk_matrix.drop(columns=['Attrition Index','Contribution_%'], inplace=True)
risk_matrix.set_index('Dimension', inplace=True)

In [ ]:
risk_matrix

,Highest Risk Segment,Employees,Exits,Attrition Rate,Largest Contributor,Contributor_Employees,Contributor_Exits,Contributor Attrition Rate
Dimension,,,,,,,,
JobRole,Sales Representative,83.0,33.0,39.76,Laboratory Technician,259.0,62.0,23.94
Tenure_Band,New Hire,215.0,75.0,34.88,New Hire,215.0,75.0,34.88
Age_Band,18-25,115.0,40.0,34.78,26-35,606.0,116.0,19.14
WorkLifeBalance,1,80.0,25.0,31.25,3,893.0,127.0,14.22
OverTime,Yes,416.0,127.0,30.53,Yes,416.0,127.0,30.53
Income_Band,Low,369.0,108.0,29.27,Low,369.0,108.0,29.27
JobLevel,1,543.0,143.0,26.34,1,543.0,143.0,26.34
EducationField,Human Resources,27.0,7.0,25.93,Life Sciences,606.0,89.0,14.69
MaritalStatus,Single,470.0,120.0,25.53,Single,470.0,120.0,25.53


In [ ]:
core_drivers = [

    'OverTime',
    'JobRole',
    'Department',
    'Age_Band',
    'Income_Band',
    'Tenure_Band',
    'JobLevel',
    'WorkLifeBalance',
    'JobSatisfaction',
    'EnvironmentSatisfaction'
]

In [ ]:
from itertools import combinations
import pandas as pd

def scan_2d(

    df,

    columns,

    min_population=20,

    target='Attrition',

    positive_class='Yes'

):

    findings = []

    for c1, c2 in combinations(columns, 2):

        temp = df.groupby(
            [c1, c2]
        ).agg(

            Employees=(
                target,
                'count'
            ),

            Attrition_Rate=(
                target,
                lambda x:
                (x == positive_class).mean() * 100
            )

        ).reset_index()

        temp = temp[
            temp['Employees'] >= min_population
        ]

        if len(temp):

            highest = temp.sort_values(
                'Attrition_Rate',
                ascending=False
            ).iloc[0]

            findings.append({

    'Dimension':
        f"{c1} × {c2}",

    'Segment':
        f"{c1}={highest[c1]} | {c2}={highest[c2]}",

    'Employees':
        highest['Employees'],

    'Attrition_Rate':
        round(
            highest['Attrition_Rate'],
            2
        )

})

    return pd.DataFrame(
        findings
    ).sort_values(
        'Attrition_Rate',
        ascending=False
    )

In [ ]:
scan_2d(df, core_drivers)

,Dimension,Segment,Employees,Attrition_Rate
14,JobRole × WorkLifeBalance,JobRole=Laboratory Technician | WorkLifeBalance=1,20.0,70.00
0,OverTime × JobRole,OverTime=Yes | JobRole=Sales Representative,24.0,66.67
25,Age_Band × Tenure_Band,Age_Band=18-25 | Tenure_Band=New Hire,37.0,64.86
2,OverTime × Age_Band,OverTime=Yes | Age_Band=18-25,37.0,62.16
3,OverTime × Income_Band,OverTime=Yes | Income_Band=Low,106.0,58.49
10,JobRole × Age_Band,JobRole=Sales Representative | Age_Band=18-25,24.0,58.33
12,JobRole × Tenure_Band,JobRole=Sales Representative | Tenure_Band=New...,30.0,56.67
4,OverTime × Tenure_Band,OverTime=Yes | Tenure_Band=New Hire,69.0,55.07
5,OverTime × JobLevel,OverTime=Yes | JobLevel=1,156.0,52.56
32,Income_Band × WorkLifeBalance,Income_Band=Low | WorkLifeBalance=1,21.0,52.38


In [ ]:
three_d_combos = [

    ('JobRole','OverTime','Tenure_Band'),

    ('Age_Band','OverTime','WorkLifeBalance'),

    ('Income_Band','OverTime','Tenure_Band'),

    ('Department','JobLevel','OverTime'),

    ('Tenure_Band','JobSatisfaction','EnvironmentSatisfaction'),

    ('Income_Band','JobLevel','StockOptionLevel')

]

In [ ]:
import pandas as pd

def scan_3d(

    df,

    combos,

    min_population=20,

    target='Attrition',

    positive_class='Yes'

):

    findings = []

    for c1, c2, c3 in combos:

        temp = df.groupby(
            [c1, c2, c3]
        ).agg(

            Employees=(
                target,
                'count'
            ),

            Attrition_Rate=(
                target,
                lambda x:
                (x == positive_class).mean() * 100
            )

        ).reset_index()

        temp = temp[
            temp['Employees'] >= min_population
        ]

        if len(temp):

            highest = temp.sort_values(
                'Attrition_Rate',
                ascending=False
            ).iloc[0]

            findings.append({

                'Dimension':
                    f"{c1} × {c2} × {c3}",

                'Segment':
                    f"{c1}={highest[c1]} | "
                    f"{c2}={highest[c2]} | "
                    f"{c3}={highest[c3]}",

                'Employees':
                    highest['Employees'],

                'Attrition_Rate':
                    round(
                        highest['Attrition_Rate'],
                        2
                    )

            })

    return pd.DataFrame(
        findings
    ).sort_values(
        'Attrition_Rate',
        ascending=False
    )

In [ ]:
scan_3d(df, three_d_combos)

,Dimension,Segment,Employees,Attrition_Rate
2,Income_Band × OverTime × Tenure_Band,Income_Band=Low | OverTime=Yes | Tenure_Band=N...,32,81.25
0,JobRole × OverTime × Tenure_Band,JobRole=Laboratory Technician | OverTime=Yes |...,20,75.00
3,Department × JobLevel × OverTime,Department=Sales | JobLevel=1 | OverTime=Yes,22,72.73
1,Age_Band × OverTime × WorkLifeBalance,Age_Band=18-25 | OverTime=Yes | WorkLifeBalance=3,23,69.57
4,Tenure_Band × JobSatisfaction × EnvironmentSat...,Tenure_Band=New Hire | JobSatisfaction=3 | Env...,23,43.48
5,Income_Band × JobLevel × StockOptionLevel,Income_Band=Low | JobLevel=1 | StockOptionLevel=0,169,41.42


In [ ]:
pd.crosstab(
    [df['JobRole'],
     df['Income_Band']],
    df['Attrition'],
    normalize='index'
) * 100

Attrition                                      No        Yes
JobRole                   Income_Band                       
Healthcare Representative Medium        95.000000   5.000000
                          High          95.454545   4.545455
                          Very High     88.888889  11.111111
Human Resources           Low           60.869565  39.130435
                          Medium        91.666667   8.333333
                          High         100.000000   0.000000
                          Very High     60.000000  40.000000
Laboratory Technician     Low           70.769231  29.230769
                          Medium        79.807692  20.192308
                          High          88.000000  12.000000
Manager                   Very High     95.098039   4.901961
Manufacturing Director    Medium        90.322581   9.677419
                          High          95.588235   4.411765
                          Very High     91.304348   8.695652
Research Director         Very High     97.500000   2.500000
Research Scientist        Low           78.289474  21.710526
                          Medium        88.990826  11.009174
                          High          93.333333   6.666667
                          Very High    100.000000   0.000000
Sales Executive           Medium        87.671233  12.328767
                          High          82.926829  17.073171
                          Very High     77.528090  22.471910
Sales Representative      Low           56.250000  43.750000
                          Medium        70.588235  29.411765
                          High         100.000000   0.000000

In [ ]:
pd.crosstab(
    [df['Income_Band'],
     df['OverTime']],
    df['Attrition'],
    normalize='index'
)*100

Attrition                    No        Yes
Income_Band OverTime                      
Low         No        82.509506  17.490494
            Yes       41.509434  58.490566
Medium      No        90.000000  10.000000
            Yes       73.958333  26.041667
High        No        93.023256   6.976744
            Yes       80.733945  19.266055
Very High   No        92.775665   7.224335
            Yes       81.904762  18.095238

In [ ]:
pd.crosstab(
    [df['Income_Band'],
     df['Tenure_Band']],
    df['Attrition'],
    normalize='index'
)*100

Attrition                        No        Yes
Income_Band Tenure_Band                       
Low         New Hire      52.678571  47.321429
            Early Career  75.247525  24.752475
            Mid Career    83.132530  16.867470
            Long Tenure   78.082192  21.917808
Medium      New Hire      72.093023  27.906977
            Early Career  83.750000  16.250000
            Mid Career    85.714286  14.285714
            Long Tenure   91.304348   8.695652
High        New Hire      81.250000  18.750000
            Early Career  89.743590  10.256410
            Mid Career    88.732394  11.267606
            Long Tenure   90.666667   9.333333
Very High   New Hire      85.714286  14.285714
            Early Career  85.714286  14.285714
            Mid Career    93.617021   6.382979
            Long Tenure   89.922481  10.077519